In [20]:
import import_ipynb
from yahoo import getSP500_data
from cpi import get_cpi
from ffer import get_dff
from unemployment import get_unrate

In [21]:
cpi = get_cpi()
fedfunds = get_dff()
unrate = get_unrate()
sp500 = getSP500_data()

In [22]:
keys = ["Month", "Year"]

def one_row_per_month_year(df):
    # Keep one record per Month/Year to avoid repeated rows after merge.
    if "Date" in df.columns:
        return df.sort_values("Date").drop_duplicates(subset=keys, keep="last")
    if "observation_date" in df.columns:
        return df.sort_values("observation_date").drop_duplicates(subset=keys, keep="last")
    return df.drop_duplicates(subset=keys, keep="last")

cpi_u = one_row_per_month_year(cpi)
fedfunds_u = one_row_per_month_year(fedfunds)
unrate_u = one_row_per_month_year(unrate)
sp500_u = one_row_per_month_year(sp500)

merged_df = (
    sp500_u.merge(cpi_u, on=keys, how="inner")
           .merge(fedfunds_u, on=keys, how="inner")
           .merge(unrate_u, on=keys, how="inner")
)

# Drop date columns that appear from source DataFrames.
drop_cols = [
    c for c in merged_df.columns
    if c.lower() in {"date", "observation_date_x", "observation_date", "observation_date_y"}
]
merged_df = merged_df.drop(columns=drop_cols, errors="ignore")

# Keep only the final columns in the requested order.

merged_df.head()

,Opening_Close,Closing_Close,Monthly_Increase,Month,Year,cpi_pct,fedfunds_rate,unemployment_rate
0,909.030029,855.700012,-53.330017,1,2003,2.778159,1.33,5.8
1,860.320007,841.150024,-19.169983,2,2003,2.695620,1.33,5.9
2,834.809998,848.179993,13.369995,3,2003,2.619135,1.38,5.9
3,858.479980,916.919983,58.440002,4,2003,2.450573,1.31,6.0
4,916.299988,963.590027,47.290039,5,2003,2.338588,1.28,6.1


In [23]:
from pathlib import Path

download_path = Path.cwd().parent / "merged.csv"
merged_df.to_csv(download_path, index=False)